# 03 · Classificação — 5 modelos binários, curva de aprendizado e multi-rótulo

- **Origem:** `experiments/classification.ipynb`
- **Faz:** o notebook principal — treina e avalia os **5 modelos binários** (Tab. 7–11, Fig. 2), a **curva de aprendizado** (Fig. 3) e o **multi-rótulo** mBERT (Fig. 4). Preserva a classe `Experiment` do autor, trocando o miolo `simpletransformers`/`apex` por HuggingFace `Trainer`.
- **Entradas:** `experiments/data/1annotator.zip`, `ToLD-BR.csv` e o OLID via `cardiffnlp/tweet_eval`.
- **Saídas:** `results/binary_*.json`, `results/preds_binary_*.csv`, `results/learning_curve.json`, `results/multilabel_*.json` e figuras em `results/figures/`. **≈ 1,5–2 h de GPU.**

> **Diff mínimo.** Cada alteração abaixo é marcada **[T]** tecnologia descontinuada, **[B]** bug ou **[A]** fidelidade ao artigo — catálogo completo em [`CHANGES.md`](CHANGES.md).

In [1]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "ToLD-BR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_ZIP  = ROOT / "experiments" / "data" / "1annotator.zip"
MAIN_CSV  = ROOT / "ToLD-BR.csv"
ALPHA_CSV = ROOT / "ToLD-BR_alpha.csv"
RESULTS   = ROOT / "reproduction" / "results"
FIGURES   = RESULTS / "figures"
RESULTS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

**Mudança #1 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | removido todo o setup do Google Colab |
| **Antes** | `!nvidia-smi`; `from google.colab import drive, files` + `drive.mount(...)` + `datasets_path="drive/My Drive/..."`; `%%writefile setup.sh` (clona/compila NVIDIA apex, pip simpletransformers/unidecode); `!sh setup.sh` |
| **Depois** | células removidas — imports padrão; dados de `DATA_ZIP`/`MAIN_CSV` e OLID via `datasets` |
| **Motivo** | apex não builda em CUDA/Python modernos e simpletransformers foi substituído por HuggingFace transformers; o Drive não existe localmente |
| **Impacto** | nenhum no resultado — apenas infraestrutura; os dados vêm de DATA_ZIP/MAIN_CSV e o OLID via datasets |

**Mudança #2 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | imports trocados de simpletransformers para HuggingFace transformers |
| **Antes** | `from simpletransformers.classification import ClassificationModel, MultiLabelClassificationModel` + logging do simpletransformers |
| **Depois** | `AutoTokenizer`, `AutoModelForSequenceClassification`, `Trainer`, `TrainingArguments`, `set_seed` + `datasets.Dataset` |
| **Motivo** | simpletransformers/apex descontinuados; o stack moderno roda em 2026. Mantidos sklearn (métricas/SVM/BoW), nltk (stopwords), unidecode (preprocessing) — todos preservados do original |
| **Impacto** | tecnologia; metodologia idêntica |

In [2]:
import io
import json
import re
import string
import zipfile

import numpy as np
import pandas as pd
import torch
import nltk
from unidecode import unidecode
from datasets import Dataset, load_dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments, set_seed)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             confusion_matrix, recall_score, precision_score,
                             roc_auc_score, multilabel_confusion_matrix,
                             hamming_loss, average_precision_score)

nltk.download("punkt")
nltk.download("stopwords")

SEED = 42

# Modelos (fieis ao artigo — NUNCA distilbert)
BR_BERT = "neuralmind/bert-base-portuguese-cased"   # BERTimbau (PT monolingue)
M_BERT  = "bert-base-multilingual-cased"            # mBERT (multilingue)

c:\Users\Pedro\AppData\Local\toldbr-repro-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Pedro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Pedro\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Binary Classification

**Mudança #3 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | miolo de `train()`/`eval()` portado de simpletransformers para HuggingFace `Trainer` |
| **Antes** | `ClassificationModel("distilbert", ...).train_model()/eval_model()/predict()`; `eval()` imprimia f1_score binário (classe positiva) |
| **Depois** | `Trainer`/`TrainingArguments`; `eval()` imprime macro-F1 (`average="macro"`). Assinaturas e métodos do autor preservados (`__load_dataset`, `__preprocess`, `describe_data`, `train`, `eval`) |
| **Motivo** | simpletransformers/apex descontinuados; macro-F1 é a métrica de comparação |
| **Impacto** | tecnologia (a métrica macro-F1 é a que o artigo reporta) |

**Mudança #4 · [A] fidelidade**

| | |
|:--|:--|
| **O quê** | modelo BERT fiel ao artigo (mBERT completo / BERTimbau) no lugar do distilbert |
| **Antes** | `model_name="distilbert"`; `pretrained_name="distilbert-base-multilingual-cased"` |
| **Depois** | `bert-base-multilingual-cased` (mBERT) e `neuralmind/bert-base-portuguese-cased` (BERTimbau, caso PT) |
| **Motivo** | fidelidade ao artigo (ÚNICA mudança que altera resultado, confirmada pelo usuário) |
| **Impacto** | alto e esperado — reproduz os números do artigo (BR-BERT 0.768, M-BERT-BR 0.756) |

**Mudança #5 · [B] bug · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | adicionado `warmup_ratio=0.06` (default do simpletransformers original) |
| **Antes** | ausente na reimplementação → o transfer colapsava na classe majoritária (macro-F1 0.349) |
| **Depois** | `warmup_ratio=0.06` nos `TrainingArguments` |
| **Motivo** | corrige a instabilidade de fine-tuning E é mais fiel ao original |
| **Impacto** | alto — transfer passa de 0.349 para 0.756 |

**Mudança #6 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | `max_seq_len` e `train_batch_size` ajustados |
| **Antes** | `max_seq_len=512`; `train_batch_size=50` (valores do artigo) |
| **Depois** | `max_seq_len=128`; `train_batch_size=32` |
| **Motivo** | tweets cabem em 128 tokens (~280 chars) e memória/desempenho nos 16 GB |
| **Impacto** | desprezível |

**Mudança #7 · [B] bug**

| | |
|:--|:--|
| **O quê** | corrigida a indentação quebrada em `__init__`/`__preprocess` |
| **Antes** | as linhas `self.train_set/self.test_set = self.__preprocess(...)` ficavam aninhadas dentro do `else` do bloco de stopwords, então só eram aplicadas no caso != multilingual |
| **Depois** | linhas desaninhadas — `__preprocess` roda sempre que `do_preprocessing=True` |
| **Motivo** | bug |
| **Impacto** | preprocessing passa a ser aplicado quando `do_preprocessing=True` (irrelevante no caminho padrão do artigo, que usa `do_preprocessing=False`) |

**Mudança #8 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | OLID carregado via `datasets` (`_load_olid`) de `cardiffnlp/tweet_eval`, config offensive, train+validation=13240 |
| **Antes** | `pd.read_csv(datasets_path+"olid-training-v1.0.tsv", sep="\t")` |
| **Depois** | `load_dataset("cardiffnlp/tweet_eval", "offensive")` (train+validation) |
| **Motivo** | o TSV vivia no Drive; datasets 4.x exige repo namespaced |
| **Impacto** | nenhum — é o mesmo OLID (label OFF→1, demais→0) |

In [3]:
def _load_olid():
    """OLID (OffensEval 2019) via cardiffnlp/tweet_eval, config offensive.
    train+validation = 13240. label: 0 nao-ofensivo, 1 ofensivo -> 'toxic'."""
    ds = load_dataset("cardiffnlp/tweet_eval", "offensive")
    parts = [ds[sp].to_pandas() for sp in ("train", "validation")]
    df = pd.concat(parts, ignore_index=True).rename(columns={"label": "toxic"})
    return df[["text", "toxic"]].dropna().reset_index(drop=True)


class Experiment():
  def __init__(
      self, bert_args=None, language="portuguese", n_annotators=1,
      balancing=None, do_preprocessing=False, data_amount=1,
      text_representation_model="bow", pretrained_name=M_BERT
  ):
    """
    language (str) -> portuguese, multilanguage ou english.
    n_annotators -> minimo de concordancia entre anotadores para considerar toxico.
    bert_args (dict) -> argumentos do modelo BERT (epochs, batch, max_len, lr, seed).
    balancing (str ou None) -> 'undersampling', 'oversampling' ou None.
    do_preprocessing (bool) -> remove stopwords, acentos, numeros, hashtags, pontuacao.
    data_amount (float) -> valor em (0,1] usado como fracao dos dados de treino.
    text_representation_model (str) -> "bert" ou "bow". Bow implica usar SVM.
    pretrained_name (str) -> checkpoint HuggingFace para o caso "bert".
    """
    self.language = language
    self.data_amount = data_amount
    self.n_annotators = n_annotators
    self.pretrained_name = pretrained_name
    self.__load_dataset(self.language)
    self.text_representation_model = text_representation_model

    if do_preprocessing:
      if self.language == "multilingual":
        stopwords = list(set([unidecode(w) for w in
                              nltk.corpus.stopwords.words("portuguese")]))
        stopwords.extend(list(set([unidecode(w) for w in
                              nltk.corpus.stopwords.words("english")])))
      else:
        stopwords = list(set([unidecode(w) for w in
                              nltk.corpus.stopwords.words(self.language)]))
      # MUDANCA #7 [B]: estas duas linhas estavam aninhadas dentro do else acima
      self.train_set = self.__preprocess(self.train_set, stopwords=stopwords)
      self.test_set = self.__preprocess(self.test_set, stopwords=stopwords)

    if balancing == "undersampling":
      negatives = self.train_set["toxic"] == 0
      positives = self.train_set["toxic"] == 1
      self.train_set = pd.concat([
          self.train_set[negatives].sample(len(self.train_set[positives]),
                                            random_state=SEED),
          self.train_set[positives]], ignore_index=True)
    elif balancing == "oversampling":
      positives = self.train_set[self.train_set["toxic"] == 1]
      self.train_set = pd.concat([self.train_set, positives], ignore_index=True)
    elif balancing is None:
      pass
    else:
      raise(NotImplementedError)

    self.describe_data()
    self.bert_args = bert_args or {}

  def __load_dataset(self, language):
    self.train_set = pd.DataFrame()
    self.test_set = pd.DataFrame()

    def _read(split):
      with zipfile.ZipFile(DATA_ZIP) as z:
        raw = z.read(f"{self.n_annotators}annotator/"
                     f"ptbr_{split}_{self.n_annotators}annotator.csv")
      df = pd.read_csv(io.BytesIO(raw))
      return df[["text", "toxic"]].dropna().reset_index(drop=True)

    # Test set (sempre o ptbr_test, independente da language)
    self.test_set = _read("test")

    if language == "portuguese" or language == "multilanguage":
      train = _read("train")
      dev_set = _read("validation")
      self.train_set = pd.concat([train, dev_set], ignore_index=True)

    if language == "english" or language == "multilanguage":
      eng_data = _load_olid()  # MUDANCA #8 [T]
      self.train_set = pd.concat([self.train_set, eng_data], ignore_index=True)

    self.train_set = self.train_set.sample(
        frac=self.data_amount, random_state=SEED).reset_index(drop=True)

  def __preprocess(self, data, stopwords):
    """Remove hashtags, numeros, pontuacao, acentos, links e stopwords."""
    df = data.copy()
    df["text"] = df["text"].apply(lambda x: re.sub("#[^ ]+", "", x))        # hashtags
    df["text"] = df["text"].apply(lambda x: re.sub(r"\d+", "", x))          # numeros
    df["text"] = df["text"].apply(lambda x: x.translate(
        str.maketrans("", "", string.punctuation)))                        # pontuacao
    df["text"] = df["text"].apply(lambda x: unidecode(x))                   # acentos
    df["text"] = df["text"].apply(lambda x: re.sub("http[^ ]+", "", x))     # links
    df["text"] = df["text"].apply(lambda x: " ".join(
        w.strip() for w in x.split() if w not in stopwords))               # stopwords
    return df

  def describe_data(self):
    """Imprime contagens e razoes do train/test set."""
    negative_count, positive_count = self.train_set["toxic"].value_counts()
    total = negative_count + positive_count
    train_ratio = len(self.train_set)/(len(self.train_set) + len(self.test_set))
    print(f"""TRAIN SET ({train_ratio:.2f})
            Total: {total}
            -------Counts-------
            Negatives: {negative_count}
            Positives: {positive_count}
            -------Ratio--------
            Negatives: {negative_count/total:.2f}
            Positives: {positive_count/total:.2f}
        """)
    negative_count, positive_count = self.test_set["toxic"].value_counts()
    total = negative_count + positive_count
    test_ratio = len(self.test_set)/(len(self.train_set) + len(self.test_set))
    print(f"""TEST SET ({test_ratio:.2f})
            Total: {total}
            -------Counts-------
            Negatives: {negative_count}
            Positives: {positive_count}
            -------Ratio--------
            Negatives: {negative_count/total:.2f}
            Positives: {positive_count/total:.2f}
        """)

  def __make_training_args(self):
    a = self.bert_args
    return TrainingArguments(
        output_dir=str(RESULTS / "_tmp" / f"bin_{a.get('manual_seed', SEED)}"),
        num_train_epochs=a.get("num_train_epochs", 3),
        per_device_train_batch_size=a.get("train_batch_size", 32),  # MUDANCA #6: 50->32
        per_device_eval_batch_size=64,
        learning_rate=a.get("learning_rate", 4e-5),
        warmup_ratio=0.06,                                          # MUDANCA #5 [B/T]
        seed=a.get("manual_seed", SEED),
        fp16=torch.cuda.is_available(),
        save_strategy="no",
        report_to=[],
        logging_steps=50,
        disable_tqdm=False,
    )

  def train(self):
    """Inicializa o treino (BoW+SVM ou BERT via HF Trainer)."""
    if self.text_representation_model == "bow":
      bow = CountVectorizer()
      bow.fit(self.train_set["text"])
      self.__X_train = bow.transform(self.train_set["text"])
      self.__y_train = self.train_set["toxic"]
      self.__X_test = bow.transform(self.test_set["text"])
      self.__y_test = self.test_set["toxic"]
      self.model = SVC(verbose=True)
      self.model.fit(self.__X_train, self.__y_train)

    elif self.text_representation_model == "bert":
      # MUDANCA #3 [T] + #4 [A]: miolo HF Trainer; mBERT/BERTimbau (nunca distilbert)
      a = self.bert_args
      max_len = a.get("max_seq_len", 128)                          # MUDANCA #6: 512->128
      self.__tok = AutoTokenizer.from_pretrained(
          self.pretrained_name, do_lower_case=a.get("do_lower_case", False))
      set_seed(a.get("manual_seed", SEED))

      def to_ds(df):
        d = df[["text", "toxic"]].rename(columns={"toxic": "labels"})
        ds = Dataset.from_pandas(d, preserve_index=False)
        keep = {"input_ids", "attention_mask", "token_type_ids", "labels"}
        ds = ds.map(lambda b: self.__tok(b["text"], truncation=True,
                                         padding="max_length", max_length=max_len),
                    batched=True)
        return ds.remove_columns([c for c in ds.column_names if c not in keep])

      self.__train_ds = to_ds(self.train_set)
      self.__test_ds = to_ds(self.test_set)
      self.model = AutoModelForSequenceClassification.from_pretrained(
          self.pretrained_name, num_labels=2)
      self.__trainer = Trainer(model=self.model, args=self.__make_training_args(),
                               train_dataset=self.__train_ds)
      self.__trainer.train()

  def eval(self):
    """Imprime accuracy, macro-F1, classification report e matriz de confusao.
    Retorna pd.DataFrame (text, y_true, y_pred, language)."""
    if self.text_representation_model == "bert":
      logits = self.__trainer.predict(self.__test_ds).predictions
      y_pred = logits.argmax(axis=-1)
      y_true = list(self.test_set["toxic"])
    else:
      y_pred = self.model.predict(self.__X_test)
      y_true = list(self.__y_test)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    result = f"(tn: {cm[0,0]} fp: {cm[0,1]} fn: {cm[1,0]} tp: {cm[1,1]})"

    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")  # MUDANCA #3: macro-F1 (era binario)
    print(f"Accuracy: {acc:.3f}\nMacro-F1: {macro:.3f}")
    print("----------------------------------------")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("----------------------------------------")
    print(result)

    results = pd.DataFrame(columns=["text", "y_true", "y_pred", "language"])
    results["y_true"] = list(y_true)
    results["y_pred"] = list(y_pred)
    results["text"] = self.test_set["text"].reset_index(drop=True)
    results["language"] = self.language
    self.results = results
    self.confusion_matrix = cm
    self.metrics = {"accuracy": float(acc), "macro_f1": float(macro),
                    "confusion_matrix": cm.tolist(),
                    "report": classification_report(y_true, y_pred,
                                                    output_dict=True, zero_division=0)}
    return self.results

  def save_results(self, name):
    """MUDANCA #9 [T]: grava json (metricas) + csv (predicoes) em RESULTS, no lugar
    de files.download. Tambem salva a matriz de confusao em FIGURES."""
    with open(RESULTS / f"{name}.json", "w", encoding="utf-8") as f:
      json.dump(self.metrics, f, indent=2, ensure_ascii=False)
    self.results.to_csv(RESULTS / f"preds_{name}.csv", index=False)
    self.__plot_confusion(name)

  def __plot_confusion(self, name):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cm = np.asarray(self.confusion_matrix)
    labels = ("nao-toxico", "toxico")
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("Predito"); ax.set_ylabel("Verdadeiro"); ax.set_title(name)
    for i in range(cm.shape[0]):
      for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im); fig.tight_layout()
    fig.savefig(FIGURES / f"cm_{name}.png", dpi=120)
    plt.close(fig)

Nota: `do_lower_case=False` e o default dos checkpoints *cased* (BERTimbau e mBERT-cased), entao passa-lo explicitamente nao altera nada — e apenas a explicitacao da convencao #4. O construtor agora aceita `pretrained_name` (ausente no original, onde o checkpoint era hardcoded `distilbert-base-multilingual-cased`); e por esse parametro que a MUDANCA #4 seleciona BR-BERT vs M-BERT.

## BERT

## BoW + SVM (Baseline)

**Mudança #10 · [B] bug**

| | |
|:--|:--|
| **O quê** | na instanciação do baseline, `train_amount=1.0` → `data_amount=1.0` |
| **Antes** | o construtor nunca teve parâmetro `train_amount` (o docstring tinha o nome errado), então a chamada quebrava com `TypeError` |
| **Depois** | `Experiment(..., data_amount=1.0, ...)` |
| **Motivo** | bug |
| **Impacto** | o baseline passa a instanciar; BoW+SVM = macro-F1 ~0.730 (artigo 0.74) |

In [4]:
# --- Modelo (i): baseline BoW + SVM (preservado do autor) ---
exp = Experiment(
    language="portuguese",
    n_annotators=1,
    do_preprocessing=False,
    balancing=None,
    data_amount=1.0,                # MUDANCA #10 [B]: era train_amount=1.0
    text_representation_model="bow",
)
exp.train()
exp.eval()
exp.save_results("binary_baseline_bow_svm")

TRAIN SET (0.90)
            Total: 18900
            -------Counts-------
            Negatives: 10617
            Positives: 8283
            -------Ratio--------
            Negatives: 0.56
            Positives: 0.44
        
TEST SET (0.10)
            Total: 2100
            -------Counts-------
            Negatives: 1128
            Positives: 972
            -------Ratio--------
            Negatives: 0.54
            Positives: 0.46
        
[LibSVM]Accuracy: 0.736
Macro-F1: 0.730
----------------------------------------
              precision    recall  f1-score   support

           0       0.73      0.82      0.77      1128
           1       0.75      0.64      0.69       972

    accuracy                           0.74      2100
   macro avg       0.74      0.73      0.73      2100
weighted avg       0.74      0.74      0.73      2100

----------------------------------------
(tn: 922 fp: 206 fn: 349 tp: 623)


**Mudança #11 · [A] fidelidade**

| | |
|:--|:--|
| **O quê** | instanciadas as 4 variantes BERT do artigo via a própria classe `Experiment` |
| **Antes** | uma única instância distilbert PT (cell-13/14/15) + células de zip/ls/load do outputs (cell-16/17/18) |
| **Depois** | (ii) BR-BERT (BERTimbau, PT); (iii) M-BERT-BR (mBERT, PT); (iv) M-BERT transfer (mBERT, ToLD-Br + OLID); (v) M-BERT zero-shot (mBERT, treino só OLID) |
| **Motivo** | produzir os 5 modelos binários do artigo (o baseline é o (i)) |
| **Impacto** | alto e esperado — reproduz as Tabelas 7-11 |

In [5]:
# Hiperparametros comuns dos 4 modelos BERT (artigo: 3 epocas, seed 42, do_lower_case=False)
BERT_ARGS = {
    "num_train_epochs": 3,
    "manual_seed": SEED,
    "do_lower_case": False,
    "train_batch_size": 32,   # MUDANCA #6: artigo 50
    "max_seq_len": 128,       # MUDANCA #6: artigo 512
    "learning_rate": 4e-5,
}

# --- Modelo (ii): BR-BERT (BERTimbau, PT monolingue) ---
exp_br = Experiment(language="portuguese", text_representation_model="bert",
                    pretrained_name=BR_BERT, bert_args=BERT_ARGS)
exp_br.train(); exp_br.eval()
exp_br.save_results("binary_br_bert")

# --- Modelo (iii): M-BERT-BR (mBERT treinado so em PT) ---
exp_mbr = Experiment(language="portuguese", text_representation_model="bert",
                     pretrained_name=M_BERT, bert_args=BERT_ARGS)
exp_mbr.train(); exp_mbr.eval()
exp_mbr.save_results("binary_mbert_br")  # gera preds_binary_mbert_br.csv (insumo da analise de erro / Tab. 12, feita pelo notebook 07 — ver nota ao fim deste notebook)

# --- Modelo (iv): M-BERT transfer (mBERT, ToLD-Br + OLID) ---
exp_tr = Experiment(language="multilanguage", text_representation_model="bert",
                    pretrained_name=M_BERT, bert_args=BERT_ARGS)
exp_tr.train(); exp_tr.eval()
exp_tr.save_results("binary_mbert_transfer")

# --- Modelo (v): M-BERT zero-shot (mBERT, treino SO no OLID em ingles) ---
exp_zs = Experiment(language="english", text_representation_model="bert",
                    pretrained_name=M_BERT, bert_args=BERT_ARGS)
exp_zs.train(); exp_zs.eval()
exp_zs.save_results("binary_mbert_zeroshot")

TRAIN SET (0.90)
            Total: 18900
            -------Counts-------
            Negatives: 10617
            Positives: 8283
            -------Ratio--------
            Negatives: 0.56
            Positives: 0.44
        
TEST SET (0.10)
            Total: 2100
            -------Counts-------
            Negatives: 1128
            Positives: 972
            -------Ratio--------
            Negatives: 0.54
            Positives: 0.46
        


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 82428.06it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
50,0.678851
100,0.563082
150,0.534154
200,0.514276
250,0.472201
300,0.484902
350,0.499246
400,0.450760
450,0.469800
500,0.459083


Accuracy: 0.766
Macro-F1: 0.765
----------------------------------------
              precision    recall  f1-score   support

           0       0.78      0.78      0.78      1128
           1       0.75      0.75      0.75       972

    accuracy                           0.77      2100
   macro avg       0.76      0.76      0.76      2100
weighted avg       0.77      0.77      0.77      2100

----------------------------------------
(tn: 884 fp: 244 fn: 247 tp: 725)
TRAIN SET (0.90)
            Total: 18900
            -------Counts-------
            Negatives: 10617
            Positives: 8283
            -------Ratio--------
            Negatives: 0.56
            Positives: 0.44
        
TEST SET (0.10)
            Total: 2100
            -------Counts-------
            Negatives: 1128
            Positives: 972
            -------Ratio--------
            Negatives: 0.54
            Positives: 0.46
        


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3814.83it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

...

Step,Training Loss
50,0.679951
100,0.656267
150,0.605465
200,0.572550
250,0.555280
300,0.538371
350,0.530022
400,0.523657
450,0.524806
500,0.538727


Accuracy: 0.752
Macro-F1: 0.752
----------------------------------------
              precision    recall  f1-score   support

           0       0.80      0.73      0.76      1128
           1       0.71      0.78      0.75       972

    accuracy                           0.75      2100
   macro avg       0.75      0.75      0.75      2100
weighted avg       0.76      0.75      0.75      2100

----------------------------------------
(tn: 819 fp: 309 fn: 211 tp: 761)
TRAIN SET (0.86)
            Total: 13240
            -------Counts-------
            Negatives: 8840
            Positives: 4400
            -------Ratio--------
            Negatives: 0.67
            Positives: 0.33
        
TEST SET (0.14)
            Total: 2100
            -------Counts-------
            Negatives: 1128
            Positives: 972
            -------Ratio--------
            Negatives: 0.54
            Positives: 0.46
        


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21967.22it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Step,Training Loss
50,0.651281
100,0.574213
150,0.524306
200,0.528287
250,0.491742
300,0.478430
350,0.487968
400,0.483701
450,0.409071
500,0.414043


Accuracy: 0.587
Macro-F1: 0.544
----------------------------------------
              precision    recall  f1-score   support

           0       0.58      0.83      0.68      1128
           1       0.61      0.30      0.41       972

    accuracy                           0.59      2100
   macro avg       0.59      0.57      0.54      2100
weighted avg       0.59      0.59      0.55      2100

----------------------------------------
(tn: 936 fp: 192 fn: 676 tp: 296)


Nota: `language="english"` em `__load_dataset` carrega **apenas** o OLID no `train_set` (sem ToLD-Br) — e exatamente o cenario zero-shot do artigo (treina em ingles, testa em PT). O `test_set` e sempre o `ptbr_test` (carregado incondicionalmente). `language="multilanguage"` carrega ToLD-Br + OLID (transfer). Reproduz o bloco binario do artigo (BR-BERT, M-BERT-BR com `save_preds`, transfer = `tox_train + olid`, zero-shot = `olid`).

## Learning Curve Experiment

**Mudança #12 · [B] bug**

| | |
|:--|:--|
| **O quê** | corrigida a variável `it` indefinida no laço da curva |
| **Antes** | `print(f"...{it}/30...")` e `np.random.seed(seeds[it])` usavam `it`, nunca inicializada (NameError na 1ª iteração; havia `it += 1` no fim sem `it = 0` antes) |
| **Depois** | usa o índice sequencial do laço (`run = 1..30`) como semente |
| **Motivo** | bug |
| **Impacto** | a curva passa a rodar |

**Mudança #13 · [T] tecnologia · [A] fidelidade**

| | |
|:--|:--|
| **O quê** | curva de aprendizado com M-BERT-BR; salva `learning_curve.json` em RESULTS para o notebook 04 |
| **Antes** | `ClassificationModel("distilbert", ...)`, 512 tokens, `n_gpu=4`, `files.download` |
| **Depois** | M-BERT-BR, 1 época, 3 repetições por fração (10%..100%), grava `RESULTS/learning_curve.json` |
| **Motivo** | fidelidade (mBERT) + tecnologia (Drive/Colab/simpletransformers) |
| **Impacto** | reproduz a Figura 3 (macro-F1 sobe de ~0.58 a ~0.73-0.75) |

In [6]:
# Curva de aprendizado: M-BERT-BR, 1 epoca, fracoes 10%..100%, 3 repeticoes/fracao.
# Curva de aprendizado. Reutiliza o pipeline da classe Experiment via data_amount.
def _read_split(split):
    with zipfile.ZipFile(DATA_ZIP) as z:
        raw = z.read(f"1annotator/ptbr_{split}_1annotator.csv")
    df = pd.read_csv(io.BytesIO(raw))
    return df[["text", "toxic"]].dropna().reset_index(drop=True)

pool = pd.concat([_read_split("train"), _read_split("validation")], ignore_index=True)
test = _read_split("test")
FRACTIONS = [round(0.1 * i, 1) for i in range(1, 11)]
N_REPS = 3

def _finetune_subset(subset, test_df, seed):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(M_BERT, do_lower_case=False)
    def to_ds(df):
        d = df[["text", "toxic"]].rename(columns={"toxic": "labels"})
        ds = Dataset.from_pandas(d, preserve_index=False)
        keep = {"input_ids", "attention_mask", "token_type_ids", "labels"}
        ds = ds.map(lambda b: tok(b["text"], truncation=True,
                                  padding="max_length", max_length=128), batched=True)
        return ds.remove_columns([c for c in ds.column_names if c not in keep])
    train_ds, test_ds = to_ds(subset), to_ds(test_df)
    model = AutoModelForSequenceClassification.from_pretrained(M_BERT, num_labels=2)
    args = TrainingArguments(
        output_dir=str(RESULTS / "_tmp" / f"lc_{seed}"),
        num_train_epochs=1, per_device_train_batch_size=32,
        per_device_eval_batch_size=64, learning_rate=4e-5, warmup_ratio=0.06,
        seed=seed, fp16=torch.cuda.is_available(), save_strategy="no",
        report_to=[], logging_steps=50, disable_tqdm=True)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds)
    trainer.train()
    return trainer.predict(test_ds).predictions.argmax(axis=-1)

curve = {}
run = 0
for frac in FRACTIONS:
    recs = {"f1_pos": [], "f1_neg": [], "recall_pos": [], "recall_neg": [],
            "precision_pos": [], "precision_neg": [], "macro_f1": []}
    for _rep in range(N_REPS):
        run += 1
        seed = run  # MUDANCA #12 [B]: 1..30, indice do laco (era `it` indefinida)
        print(f"-------------{run}/30 (frac={frac})-------------", flush=True)
        np.random.seed(seed)
        idx = np.random.randint(0, len(pool), int(len(pool) * frac))
        subset = pool.iloc[idx].reset_index(drop=True)
        y_pred = _finetune_subset(subset, test, seed)
        y_true = test["toxic"].values
        rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        recs["macro_f1"].append(rep["macro avg"]["f1-score"])
        recs["f1_neg"].append(rep["0"]["f1-score"]); recs["f1_pos"].append(rep["1"]["f1-score"])
        recs["recall_neg"].append(rep["0"]["recall"]); recs["recall_pos"].append(rep["1"]["recall"])
        recs["precision_neg"].append(rep["0"]["precision"]); recs["precision_pos"].append(rep["1"]["precision"])
    curve[str(frac)] = {k: float(np.mean(v)) for k, v in recs.items()}
    with open(RESULTS / "learning_curve.json", "w", encoding="utf-8") as f:
        json.dump(curve, f, indent=2)  # MUDANCA #13: parcial a cada fracao (era files.download)

# Figura da curva (Figura 3)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fracs = [float(f) for f in curve]
sizes = [int(len(pool) * f) for f in fracs]
fig, ax = plt.subplots(figsize=(7, 5))
for key, lbl in [("recall_pos", "recall+"), ("recall_neg", "recall-"),
                 ("precision_pos", "precision+"), ("precision_neg", "precision-")]:
    ax.plot(sizes, [curve[str(f)][key] for f in fracs], marker="o", label=lbl)
ax.set_xlabel("# exemplos de treino"); ax.set_ylabel("score")
ax.set_title("Curva de aprendizado (M-BERT-BR)"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "learning_curve.png", dpi=120)
plt.close(fig)
print("learning_curve.json e learning_curve.png salvos em RESULTS/FIGURES")

-------------1/30 (frac=0.1)-------------


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16512.68it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

{'loss': '0.6766', 'grad_norm': '3.077', 'learning_rate': '8.571e-06', 'epoch': '0.8333'}
{'train_runtime': '4.577', 'train_samples_per_second': '413', 'train_steps_per_second': '13.11', 'train_loss': '0.6663', 'epoch': '1'}
-------------2/30 (frac=0.1)-------------


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25185.31it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

{'loss': '0.6872', 'grad_norm': '2.551', 'learning_rate': '8.571e-06', 'epoch': '0.8333'}
{'train_runtime': '4.703', 'train_samples_per_second': '401.9', 'train_steps_per_second': '12.76', 'train_loss': '0.6883', 'epoch': '1'}
-------------3/30 (frac=0.1)-------------


...

{'loss': '0.6953', 'grad_norm': '5.086', 'learning_rate': '3.935e-05', 'epoch': '0.0846'}
{'loss': '0.6256', 'grad_norm': '6.675', 'learning_rate': '3.575e-05', 'epoch': '0.1692'}
{'loss': '0.5627', 'grad_norm': '4.394', 'learning_rate': '3.214e-05', 'epoch': '0.2538'}
{'loss': '0.5525', 'grad_norm': '4.869', 'learning_rate': '2.854e-05', 'epoch': '0.3384'}
{'loss': '0.5088', 'grad_norm': '3.341', 'learning_rate': '2.494e-05', 'epoch': '0.423'}
{'loss': '0.5246', 'grad_norm': '3.614', 'learning_rate': '2.133e-05', 'epoch': '0.5076'}
{'loss': '0.4822', 'grad_norm': '4.692', 'learning_rate': '1.773e-05', 'epoch': '0.5922'}
{'loss': '0.4861', 'grad_norm': '3.787', 'learning_rate': '1.413e-05', 'epoch': '0.6768'}
{'loss': '0.4503', 'grad_norm': '5.9', 'learning_rate': '1.052e-05', 'epoch': '0.7614'}
{'loss': '0.4619', 'grad_norm': '5.651', 'learning_rate': '6.919e-06', 'epoch': '0.846'}
{'loss': '0.4619', 'grad_norm': '3.462', 'learning_rate': '3.315e-06', 'epoch': '0.9306'}
{'train_runtim

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20089.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

{'loss': '0.6929', 'grad_norm': '6.655', 'learning_rate': '3.914e-05', 'epoch': '0.0846'}
{'loss': '0.637', 'grad_norm': '5.842', 'learning_rate': '3.553e-05', 'epoch': '0.1692'}
{'loss': '0.5745', 'grad_norm': '3.898', 'learning_rate': '3.193e-05', 'epoch': '0.2538'}
{'loss': '0.5436', 'grad_norm': '6.19', 'learning_rate': '2.832e-05', 'epoch': '0.3384'}
{'loss': '0.5322', 'grad_norm': '4.509', 'learning_rate': '2.472e-05', 'epoch': '0.423'}
{'loss': '0.5054', 'grad_norm': '13.07', 'learning_rate': '2.119e-05', 'epoch': '0.5076'}
{'loss': '0.4802', 'grad_norm': '5.962', 'learning_rate': '1.759e-05', 'epoch': '0.5922'}
{'loss': '0.48', 'grad_norm': '3.781', 'learning_rate': '1.398e-05', 'epoch': '0.6768'}
{'loss': '0.4694', 'grad_norm': '5.017', 'learning_rate': '1.038e-05', 'epoch': '0.7614'}
{'loss': '0.4559', 'grad_norm': '5.655', 'learning_rate': '6.775e-06', 'epoch': '0.846'}
{'loss': '0.4406', 'grad_norm': '7.671', 'learning_rate': '3.171e-06', 'epoch': '0.9306'}
{'train_runtime'

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12459.94it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

{'loss': '0.6961', 'grad_norm': '2.221', 'learning_rate': '3.921e-05', 'epoch': '0.0846'}
{'loss': '0.6405', 'grad_norm': '3.46', 'learning_rate': '3.56e-05', 'epoch': '0.1692'}
{'loss': '0.5622', 'grad_norm': '4.696', 'learning_rate': '3.2e-05', 'epoch': '0.2538'}
{'loss': '0.551', 'grad_norm': '4.233', 'learning_rate': '2.84e-05', 'epoch': '0.3384'}
{'loss': '0.5063', 'grad_norm': '7.169', 'learning_rate': '2.479e-05', 'epoch': '0.423'}
{'loss': '0.5042', 'grad_norm': '4.085', 'learning_rate': '2.119e-05', 'epoch': '0.5076'}
{'loss': '0.4834', 'grad_norm': '3.892', 'learning_rate': '1.759e-05', 'epoch': '0.5922'}
{'loss': '0.4576', 'grad_norm': '5.583', 'learning_rate': '1.398e-05', 'epoch': '0.6768'}
{'loss': '0.4544', 'grad_norm': '4.779', 'learning_rate': '1.038e-05', 'epoch': '0.7614'}
{'loss': '0.464', 'grad_norm': '5.804', 'learning_rate': '6.775e-06', 'epoch': '0.846'}
{'loss': '0.4356', 'grad_norm': '6.925', 'learning_rate': '3.171e-06', 'epoch': '0.9306'}
{'train_runtime': '

# Multi-label Classification

**Mudança #14 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | dados multi-rótulo lidos de `MAIN_CSV` (ToLD-BR.csv) binarizando os 6 rótulos (valor>0 → 1) |
| **Antes** | `pd.read_csv(datasets_path+"1annotator/multilabel_grouped.csv")` |
| **Depois** | `pd.read_csv(MAIN_CSV)` + `(data[LABELS] > 0).astype(int)` |
| **Motivo** | o arquivo vivia no Drive; ToLD-BR.csv local já contém as 6 categorias finas |
| **Impacto** | nenhum — mesmos rótulos (homophobia, obscene, insult, racism, misogyny, xenophobia) |

In [7]:
LABELS = ["homophobia", "obscene", "insult", "racism", "misogyny", "xenophobia"]
multilabel_data = pd.read_csv(MAIN_CSV)
multilabel_data[LABELS] = (multilabel_data[LABELS] > 0).astype(int)
multilabel_data = multilabel_data[["text"] + LABELS].dropna().reset_index(drop=True)
print(multilabel_data.shape)  # esperado: (21000, 7)

(21000, 7)


## BoW + SVM (Baseline)

**Mudança #15 · [B] bug**

| | |
|:--|:--|
| **O quê** | corrigida a variável `score` indefinida no baseline multi-rótulo |
| **Antes** | `multilabel_baseline[label] = {..., "score": score, ...}` usava `score`, nunca atribuída (NameError); saída era `to_csv` local |
| **Depois** | Binary Relevance com `hamming_loss`/`average_precision` reais + matrizes por rótulo, salvos em `RESULTS` (json) |
| **Motivo** | bug |
| **Impacto** | o baseline passa a rodar; calculamos AP macro. Reproduz Hamming ~0.069 / AP ~0.209 |

In [8]:
# Baseline multi-rotulo = Binary Relevance (um SVM BoW por rotulo). Espelha
# Baseline multi-rotulo: Binary Relevance (1 SVC por rotulo) + metricas (hamming/AP).
train_ml, test_ml = train_test_split(multilabel_data, train_size=0.9, random_state=SEED)
train_ml = train_ml.reset_index(drop=True); test_ml = test_ml.reset_index(drop=True)
y_true_ml = test_ml[LABELS].values

base_pred = np.zeros((len(test_ml), len(LABELS)), dtype=int)
for j, label in enumerate(LABELS):
    print(label, flush=True)
    bow = CountVectorizer()
    x_train = bow.fit_transform(train_ml["text"])
    x_test = bow.transform(test_ml["text"])
    clf = SVC()
    clf.fit(x_train, train_ml[label])
    base_pred[:, j] = clf.predict(x_test)
    print(confusion_matrix(y_true_ml[:, j], base_pred[:, j], labels=[0, 1]))

base_metrics = {
    "hamming_loss": float(hamming_loss(y_true_ml, base_pred)),
    "average_precision": float(average_precision_score(y_true_ml, base_pred)),
    "per_label_confusion": [c.tolist()
                            for c in multilabel_confusion_matrix(y_true_ml, base_pred)],
}
with open(RESULTS / "multilabel_baseline_bow_svm.json", "w", encoding="utf-8") as f:
    json.dump(base_metrics, f, indent=2, ensure_ascii=False)
print(f"baseline: hamming={base_metrics['hamming_loss']:.3f} "
      f"AP={base_metrics['average_precision']:.3f}")

homophobia
[[2071    3]
 [  20    6]]
obscene
[[1301  167]
 [ 263  369]]
insult
[[1648   28]
 [ 322  102]]
racism
[[2089    0]
 [  11    0]]
misogyny
[[2056    1]
 [  34    9]]
xenophobia
[[2081    0]
 [  19    0]]
baseline: hamming=0.069 AP=0.209


## BERT

**Mudança #16 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | `mBERT` multi-rótulo portado para HuggingFace `Trainer` (`problem_type="multi_label_classification"`) |
| **Antes** | `MultiLabelClassificationModel(model_type="bert", model_name="bert-base-multilingual-cased", num_labels=6)` + `train_model(...batch=8, seq=512...)` + eval/predict + `to_csv`/`files.download` |
| **Depois** | `Trainer` com `problem_type="multi_label_classification"`; mBERT preservado; batch=16, max_len=128, 3 épocas, limiar 0.5 |
| **Motivo** | simpletransformers/apex descontinuados. PRESERVADO o modelo do artigo (mBERT completo) |
| **Impacto** | tecnologia; reproduz Hamming ~0.067 / AP ~0.276 (ver ressalva 8.5 do REPORT: AP saiu acima do artigo 0.19, trade-off precisão/recall) |

In [9]:
# mBERT multi-rotulo (HF Trainer).
# Reutiliza o mesmo split (train_ml/test_ml/y_true_ml) do baseline acima.
KEEP = {"input_ids", "attention_mask", "token_type_ids", "labels"}
set_seed(SEED)
tok_ml = AutoTokenizer.from_pretrained(M_BERT, do_lower_case=False)

def to_ds_ml(df):
    d = df[["text"]].copy()
    d["labels"] = df[LABELS].values.astype("float32").tolist()
    ds = Dataset.from_pandas(d, preserve_index=False)
    ds = ds.map(lambda b: tok_ml(b["text"], truncation=True,
                                 padding="max_length", max_length=128), batched=True)
    return ds.remove_columns([c for c in ds.column_names if c not in KEEP])

train_ds_ml, test_ds_ml = to_ds_ml(train_ml), to_ds_ml(test_ml)
model_ml = AutoModelForSequenceClassification.from_pretrained(
    M_BERT, num_labels=len(LABELS), problem_type="multi_label_classification")
args_ml = TrainingArguments(
    output_dir=str(RESULTS / "_tmp" / "ml_42"),
    num_train_epochs=3, per_device_train_batch_size=16,  # multi-rotulo: batch 16
    per_device_eval_batch_size=64, learning_rate=4e-5, warmup_ratio=0.06,
    seed=SEED, fp16=torch.cuda.is_available(), save_strategy="no",
    report_to=[], logging_steps=50, disable_tqdm=False)
trainer_ml = Trainer(model=model_ml, args=args_ml, train_dataset=train_ds_ml)
trainer_ml.train()

logits = trainer_ml.predict(test_ds_ml).predictions
probs = 1.0 / (1.0 + np.exp(-logits))
y_pred_ml = (probs >= 0.5).astype(int)  # limiar 0.5 (artigo)

bert_metrics = {
    "hamming_loss": float(hamming_loss(y_true_ml, y_pred_ml)),
    "average_precision": float(average_precision_score(y_true_ml, y_pred_ml)),
    "per_label_confusion": [c.tolist()
                            for c in multilabel_confusion_matrix(y_true_ml, y_pred_ml)],
}
with open(RESULTS / "multilabel_mbert.json", "w", encoding="utf-8") as f:
    json.dump(bert_metrics, f, indent=2, ensure_ascii=False)
print(f"mbert: hamming={bert_metrics['hamming_loss']:.3f} "
      f"AP={bert_metrics['average_precision']:.3f}")

# Matrizes por rotulo (Figura 4) — uma figura por rotulo em FIGURES
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
for label, cm in zip(LABELS, multilabel_confusion_matrix(y_true_ml, y_pred_ml)):
    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["neg", "pos"]); ax.set_yticks([0, 1], ["neg", "pos"])
    ax.set_xlabel("Predito"); ax.set_ylabel("Verdadeiro"); ax.set_title(label)
    for i in range(2):
        for k in range(2):
            ax.text(k, i, str(cm[i, k]), ha="center", va="center")
    fig.tight_layout(); fig.savefig(FIGURES / f"cm_multilabel_{label}.png", dpi=120)
    plt.close(fig)
print("multilabel_mbert.json e cm_multilabel_*.png salvos")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19339.34it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Step,Training Loss
50,0.610642
100,0.311213
150,0.242746
200,0.229067
250,0.225997
300,0.229984
350,0.218024
400,0.235831
450,0.226341
500,0.209272


mbert: hamming=0.067 AP=0.276
multilabel_mbert.json e cm_multilabel_*.png salvos
